![Platform+](images/banner.png)

# LAB 1: NAVIGATE + CREATE OBJECTS
_Hands-on Snowflake for data practitioners_

> **Persona track: Data Analyst** — same lab, your angle.

👉 **In this lesson, we'll find your way around Snowflake and create your first objects.**

📚 **By the end you'll be able to:**
- Set and confirm your query context
- Explore the object hierarchy
- Create your own schema, table, and row in your own database

**Time:** ~20 min
*Source data lives in the shared, read-only `PLATPLUS_DB`; everything you create lives in your own database, warehouse, and role — each named after your username (e.g. `USER_DB` / `USER_WH` / `USER_RL`).*

---
👉 **Connecting to a compute service (first time):**

> 💡 **Read all four steps before you click Connect.** The connect dialog opens on top of the notebook and covers these instructions while it's open — skim them first; to re-read, cancel the dialog or scroll the notebook down a little before reopening it.

1. Look at the top bar of the notebook — if no service is connected, click **Connect**.
2. In the connect dialog, either pick an existing service or fill in the details and click **Create and connect** at the bottom to provision one (this can take up to a minute or two).
3. **Wait until the status shows Connected (green).** You can't set your context or run any cell until it does.
4. Now set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.) Run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.

> 💡 **Opening the documentation links:** Throughout this course, doc links open in the *same* tab and can hit a Snowsight "Application error." To read a linked page, **right-click the link and choose "Open link in new tab"** (or Cmd/Ctrl-click it).

## Session Variables
The cell below calls `get_active_session()` — a Snowpark function that gives Python the **same Snowflake session your SQL cells use**. From it we read your username and build a few variable *names* (`my_db`, `my_wh`, `my_role`) that follow the course naming convention.

> ⚠️ At this point these are just Python strings — building them doesn't activate anything. The **next cell (SQL)** is what actually runs `USE ROLE` / `USE WAREHOUSE` to set your session context, so the role/warehouse picker at the top-right won't update until that SQL runs.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)

## Set Up Your Workspace
First, set up your **personal workspace**. The cell below switches to your own role, creates and uses your own warehouse, and creates your own database for what you'll build in this lab. Every object is named after your username, so everyone in the account stays out of each other's way.

In [ ]:
%%sql -r dataframe_1
-- Set your personal context: your role + your own warehouse
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
-- Your personal database + the schema(s) this lab writes to
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.SANDBOX;
CREATE SCHEMA IF NOT EXISTS {{my_db}}.STAGING;
-- Read from the shared, read-only course data
USE DATABASE PLATPLUS_DB;
USE SCHEMA ROASTERY_SALES;

## 1 · Where Am I?
Every query runs in a **context**: role · warehouse · database · schema. You just set yours in the `provision_workspace` cell above — let's confirm it.

These are **context functions** — each returns one piece of your current session: [`CURRENT_USER()`](https://docs.snowflake.com/en/sql-reference/functions/current_user), [`CURRENT_ROLE()`](https://docs.snowflake.com/en/sql-reference/functions/current_role), [`CURRENT_WAREHOUSE()`](https://docs.snowflake.com/en/sql-reference/functions/current_warehouse), [`CURRENT_DATABASE()`](https://docs.snowflake.com/en/sql-reference/functions/current_database), and [`CURRENT_SCHEMA()`](https://docs.snowflake.com/en/sql-reference/functions/current_schema). They're handy any time you're unsure where a query will run. See the [context functions reference](https://docs.snowflake.com/en/sql-reference/functions-context).

> 💡 **Why this matters for analysts:** Context errors are the #1 reason ad-hoc queries return wrong results or fail silently — knowing your context means trusting your output.

👉 Run the cell to see yours.

In [ ]:
%%sql -r dataframe_2
-- Confirm your context
SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE(), CURRENT_SCHEMA();

> 🔎 **Review the output.** You should see your personal role (ends in `_RL`) and your own warehouse, pointed at `PLATPLUS_DB.ROASTERY_SALES` for reading. You *read* from the shared data here, but *build* in your own database.

## 2 · What's Here?
Objects live in a hierarchy: **account → database → schema → table/view**. Explore it with [`SHOW`](https://docs.snowflake.com/en/sql-reference/sql/show) commands.

👉 Run the next two cells.

In [ ]:
%%sql -r dataframe_3
-- The schemas (folders of objects) inside the shared database
SHOW SCHEMAS IN DATABASE PLATPLUS_DB;

In [ ]:
%%sql -r dataframe_4
-- The tables inside the ROASTERY_SALES schema
SHOW TABLES IN SCHEMA PLATPLUS_DB.ROASTERY_SALES;

Now peek at the actual rows. You'll notice `STORES` is short — one row per café — while `ORDERS` is long, a row for every purchase. 👉 Run each cell and look at the columns.

In [ ]:
%%sql -r dataframe_5
-- STORES: one row per Roastery & Co. location (read-only shared data)
SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.STORES;

In [ ]:
%%sql -r dataframe_6
-- ORDERS: one row per order (limit to 10 just to peek)
SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS LIMIT 10;

> 🔎 **Look at the columns in both tables.** Notice that `ORDERS` has a `STORE_ID`, and `STORES` also has a `STORE_ID`. That shared column is the **link** between the two tables.
>
> `ORDERS` tells you *how much* was sold, but only as a store **id** number. `STORES` tells you each store's **name** and city. Neither table has the whole picture on its own.
>
> For anyone newer to SQL: combining two tables on a shared column like this is called a **join** — it lets you answer a question that spans both, e.g. *"total sales by store **name**"*. You'll use a join in the next topic; for now, just spot the connection.
>
> **Your turn to think:** which column links `ORDERS` to `STORES`?

### See it. 📊
You don't need Python to chart in a notebook. Run the SQL below, then use the **Chart** button above the result to turn the table into a bar chart — no code required.

> 💡 **Why this matters for analysts:** Quick visuals in-notebook let you validate data shape and spot outliers before building a polished dashboard.

In [ ]:
%%sql -r dataframe_7
-- Sales by store — run this, then click the Chart toggle above the result to see it as a bar chart
SELECT store_name, total_sales
FROM PLATPLUS_DB.ROASTERY_SALES.SALES_BY_STORE
ORDER BY total_sales DESC;

> 🔎 **Want more chart types?** The same result can be plotted in **Python** with Plotly, which offers a much richer variety of visualizations than the built-in Chart toggle. Run the next cell to see the same data as a Plotly bar chart.

In [ ]:
import plotly.express as px
from snowflake.snowpark.context import get_active_session
session = get_active_session()
df = session.sql("""SELECT store_name, total_sales FROM PLATPLUS_DB.ROASTERY_SALES.SALES_BY_STORE ORDER BY total_sales DESC""").to_pandas()
fig = px.bar(df, x="STORE_NAME", y="TOTAL_SALES", title="Sales by store",
             color_discrete_sequence=["#29B5E8"])
fig.update_layout(xaxis_title="Store Name", yaxis_title="Total Sales")
fig

## 3 · Make Something
Now build something of your own in **your own** database — a personal **coffee tasting log**. You own what you create here. See [`CREATE TABLE`](https://docs.snowflake.com/en/sql-reference/sql/create-table).

👉 Run the cell to create the table and your first tasting note.

> 📌 **Note:** You can't break the shared `PLATPLUS_DB` — it's read-only to you. Your own database (named after your username) is your space to experiment in.

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE TABLE {{my_db}}.SANDBOX.MY_TASTING_NOTES (
  tasting_id NUMBER, coffee STRING, rating NUMBER, notes STRING, tasted_ts TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP());
INSERT INTO {{my_db}}.SANDBOX.MY_TASTING_NOTES (tasting_id, coffee, rating, notes)
  VALUES (1, 'Ethiopia Yirgacheffe', 5, 'Bright and floral — my first object on Snowflake');

> 👉 **Your turn.** Make this table yours — edit the values below to a coffee *you* like and a rating from 1–5, then run the cell.

In [ ]:
%%sql -r dataframe_9
-- YOUR TURN: change the coffee, rating (1-5), and note to your own, then run
INSERT INTO {{my_db}}.SANDBOX.MY_TASTING_NOTES (tasting_id, coffee, rating, notes)
  VALUES (2, 'YOUR COFFEE HERE', 5, 'What did you think of it?');

Next, read your table back to confirm both rows landed — the seeded example plus the one you just added.

In [ ]:
%%sql -r dataframe_10
-- See both rows: the seeded example and the one you just added
SELECT * FROM {{my_db}}.SANDBOX.MY_TASTING_NOTES ORDER BY tasting_id;

> 🔎 **Review the output.** Two rows now — the example plus your own. You created a table and inserted your own data into it.

### Your angle — Data Analyst.
Find the **view** that's already built for you and use it instead of writing joins.

> 💡 **Why this matters for analysts:** Views give you curated, join-free data — the analyst's fast path to reliable metrics without needing to understand every underlying table.

👉 Run the next cell to list the views in the schema.

In [ ]:
%%sql -r dataframe_11
-- Views are saved queries; see what's already built
SHOW VIEWS IN SCHEMA PLATPLUS_DB.ROASTERY_SALES;

> 🔎 In the output above, find the **`text`** column for `SALES_BY_STORE` — it holds the view's defining SQL. Read which tables it selects `FROM` and `JOIN`s, then answer the check below.

### Knowledge check 🧠
Check your understanding with interactive quiz questions below. Each quiz cell has a widget that presents a multiple‑choice question about Snowflake functionality.

**Instructions:**
1. Hover your cursor over the notebook cell to reveal additional controls.
1. Click the ▶️ **Play button** on the right side of each quiz cell to run it.
1. Select your answer from the options provided.
1. Review the feedback before moving on.

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q1")

Now put that view to work. `SALES_BY_STORE` is built from the two base tables you identified in the quiz — run it to see total sales per store, no JOIN required.

In [ ]:
%%sql -r dataframe_12
SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.SALES_BY_STORE ORDER BY total_sales DESC;

> 🔎 **Review the output.** Which store leads on total sales? You got the answer without writing a single JOIN.

### Break it, then fix it. 🔧
Errors are part of the job — the real skill is *reading* them. The next cell has a deliberate mistake: a column name that doesn't exist. 👉 Run it, read the error, then write a corrected query in the empty cell below.

> 💡 **Why this matters for analysts:** Reading error messages saves you from filing tickets — most failures are typos or context issues you can fix yourself in seconds.

> ⚠️ If CoCo offers a **Fix Error** suggestion (the ✨ sparkle), ignore it for now — read the error yourself and write the corrected query by hand. That's the skill this step builds.

In [ ]:
%%sql -r dataframe_13
-- This FAILS on purpose — one column name is wrong
SELECT store_name, total_sale
FROM PLATPLUS_DB.ROASTERY_SALES.SALES_BY_STORE
ORDER BY total_sale DESC;

> 🔎 Read the error — Snowflake names the *invalid identifier*. Check the real column names in the `SALES_BY_STORE` output above, then fix the query below and run it.

In [ ]:
%%sql -r dataframe_14
-- YOUR TURN: write the corrected query here, then run

## Check Your Work
Run the next cell to **check your work**. It queries your own database to confirm you created the objects and rows this lab asked for, and prints a ✅ / ⛔ result for each — no need to read the code.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
checks = [
    ('Your SANDBOX.MY_TASTING_NOTES has at least 2 tasting rows', f'SELECT COUNT(*) FROM {my_db}.SANDBOX.MY_TASTING_NOTES', 2),
]
passed = 0
for label, sql, need in checks:
    try:
        n = session.sql(sql).collect()[0][0]
        ok = n >= need
    except Exception:
        n, ok = None, False
    passed += 1 if ok else 0
    icon = '✅' if ok else '⛔'
    detail = f'(found {n})' if n is not None else '(not found)'
    print(f'{icon}  {label}  {detail}')
print('-' * 52)
if passed == len(checks):
    print(f'🎉 ALL {len(checks)} CHECKS PASSED — lab complete! Nice work.')
else:
    print(f'⚠️  {passed}/{len(checks)} passed — re-run the cells above for any ⛔ item, then run this cell again.')

### Check your understanding 🧠
A few questions on this lab's key themes. Run each quiz cell below, choose an answer, and review the feedback.

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q3")

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q4")

### Key takeaways. ❄️
- Context = role · warehouse · database · schema
- Hierarchy: account → database → schema → table/view
- You own objects you create in your own database; PLATPLUS_DB is read-only

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.